# RAG 11 — Les représentations convergent-elles ? Mesurer l'alignement cross-modal

**Navigation** : [Index](README.md) | [<< Précédent](10-RAG-Visuel-Embedding-Conjoint.ipynb)

Le [10](10-RAG-Visuel-Embedding-Conjoint.ipynb) a construit l'espace **joint** SigLIP — deux tours, un espace vectoriel — et l'a utilisé pour retrouver des pages par leur image. Ce carnet pose la question juste en dessous : *pourquoi* cet espace partagé fonctionne-t-il, et jusqu'où ce phénomène est-il **général** ?

« The Platonic Representation Hypothesis » (Huh, Cheung, Wang, Isola — MIT, ICML 2024, [arXiv:2405.07987](https://arxiv.org/abs/2405.07987)) soutient que les représentations internes des réseaux profonds **convergent** : vers un espace commun au fil de l'échelle, à travers architectures, objectifs d'entraînement — et même **modalités** (vision ↔ langue). Le papier mesure cette convergence empiriquement par le taux de **voisins mutuels** (mutual nearest-neighbors) sur des paires appariées : pour chaque image, sa légende est-elle son plus proche voisin dans l'autre modalité — et réciproquement ?

Ce carnet apporte au corpus **la mesure** que le [10] n'apportait pas : il n'y avait ni mutual NN, ni CKA cross-modale, ni lecture de la **pente** de l'alignement avec la compétence. Six temps :

1. **Corpus** : 72 paires image/légende déterministes (formes synthétiques PIL, graine fixée) — un appariement exact, sans bruit d'étiquetage ;
2. **Espace joint** : SigLIP encode images et légendes dans un même espace ; le taux de voisins mutuels en est le **score d'alignement** ;
3. **Contrôle** : on brouille l'appariement — la métrique réagit, elle est donc vivante ; mais le score absolu ne se lit jamais seul (§6 du papier) ;
4. **Pente intra-modale** : l'alignement (CKA) d'encodeurs texte autonomes avec la tour texte de SigLIP croît-il avec leur taille ? (`e5-small` → `e5-base`, plancher aléatoire en référence) ;
5. **Test cross-modal** : deux encodeurs **indépendants** — `DINOv2` (images) + `e5` (texte), jamais alignés l'un à l'autre — contre la paire jointe SigLIP : l'alignement est-il automatique ?
6. **Plafond d'information** : légendes longue / moyenne / courte — l'alignement est-il **plafonné par l'information mutuelle** du signal, pas par la qualité du modèle ?

Dépendances : `pip install torch transformers sentencepiece pillow numpy`. Les quatre modèles (`google/siglip-base-patch16-256-multilingual`, `facebook/dinov2-base`, `intfloat/multilingual-e5-base`, `intfloat/multilingual-e5-small`) sont publics ; ils sont téléchargés au premier chargement puis mis en cache par Hugging Face. Aucune donnée n'est committée : les images vivent en mémoire, seules les sorties mesurées font partie du livrable.

## Plan

1. Le corpus : 72 paires image/légende déterministes
2. L'espace joint : SigLIP et le score d'alignement par voisins mutuels
3. Le contrôle : une métrique vivante, un chiffre isolé muet
4. La pente intra-modale : l'alignement croît avec la taille de l'encodeur
5. Le test cross-modal : deux encodeurs jamais alignés
6. Le plafond d'information mutuelle : légendes longue / moyenne / courte
7. Exercices et conclusion

In [1]:
import numpy as np
import torch
from PIL import Image, ImageDraw

from transformers import AutoModel, AutoProcessor, AutoImageProcessor, AutoTokenizer

SEED = 20261024
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Peripherique :", DEVICE)

Peripherique : cuda


## 1. Le corpus : 72 paires image/légende déterministes

Le papier mesure la convergence sur des paires **réelles** (WIT : photos Wikipédia et leurs légendes). Ce carnet choisit d'abord le terrain **contrôlé** : 72 combinaisons *forme × couleur × nombre × taille* (3 × 4 × 3 × 2 = 72), une image pour chacune, générées par un générateur seédé. Chaque légende décrit **exactement** ce que porte l'image — l'appariement est parfait par construction, donc ce que la mesure isole, c'est le modèle, pas le bruit d'étiquetage. La contrepartie est assumée : des formes colorées sont un terrain **facile** (le §6 reviendra dessus).

In [2]:
FORMES = ["cercle", "carre", "triangle"]
NOM_FORME = {"cercle": "cercle", "carre": "carré", "triangle": "triangle"}
COULEURS = ["rouge", "bleu", "vert", "jaune"]
CODE_COULEUR = {"rouge": "#d32f2f", "bleu": "#1565c0",
                "vert": "#2e7d32", "jaune": "#f9a825"}
NOMBRES = {1: "un", 2: "deux", 3: "trois"}
TAILLES = {"petit": 30, "grand": 62}

def pluriel(n, mot):
    return mot if n == 1 else mot + "s"

def legende(n, forme, couleur, taille):
    return (f"{NOMBRES[n]} {pluriel(n, NOM_FORME[forme])} "
            f"{pluriel(n, couleur)} {pluriel(n, taille)}")

def dessiner(n, forme, couleur, taille, rng, cote=256):
    """n formes de la couleur/taille donnees, positions tirees du rng commun."""
    img = Image.new("RGB", (cote, cote), "white")
    d = ImageDraw.Draw(img)
    r = TAILLES[taille]
    col = CODE_COULEUR[couleur]
    for _ in range(n):
        cx = int(rng.integers(r + 6, cote - r - 6))
        cy = int(rng.integers(r + 6, cote - r - 6))
        if forme == "cercle":
            d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=col)
        elif forme == "carre":
            d.rectangle([cx - r, cy - r, cx + r, cy + r], fill=col)
        else:
            d.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=col)
    return img

rng = np.random.default_rng(SEED)
corpus = []
for n in NOMBRES:
    for forme in FORMES:
        for couleur in COULEURS:
            for taille in TAILLES:
                corpus.append({"n": n, "forme": forme, "couleur": couleur,
                               "taille": taille,
                               "img": dessiner(n, forme, couleur, taille, rng),
                               "cap": legende(n, forme, couleur, taille)})

N = len(corpus)
print(f"Corpus : {N} paires image/legende (une par combinaison)")
for ex in corpus[:3] + corpus[-2:]:
    print("   ", repr(ex["cap"]))

Corpus : 72 paires image/legende (une par combinaison)
    'un cercle rouge petit'
    'un cercle rouge grand'
    'un cercle bleu petit'
    'trois triangles jaunes petits'
    'trois triangles jaunes grands'


### Lecture : ce que ce terrain contrôlé mesure — et ce qu'il ne mesure pas

Le terrain contrôlé a un seul but : rendre l'appariement **exact** (une légende par image, sans bruit d'étiquetage) pour que la mesure isole le **modèle**. Mais il a un effet de bord qu'il faut nommer d'emblée, et que la sortie du §2 rendra tangible : ces formes géométriques de couleurs plates et ces légendes très courtes sont **hors distribution** pour des modèles entraînés sur des images et des textes naturels. Un terrain « facile » pour un humain n'est pas « facile » pour SigLIP — qui n'a jamais vu un carré rouge plat sur fond blanc.

On ne lira donc jamais le score en valeur absolue. La question n'est pas « quel est le score ? » mais **comment il bouge** quand on déplace un paramètre à la fois — la compétence de l'encodeur (§4), la dépendance entre encodeurs (§5), l'information portée par la légende (§6). C'est dans ces contrastes, pas dans le niveau, que se lit ce que la mesure apporte.

## 2. L'espace joint : SigLIP et le taux de voisins mutuels

SigLIP ([10](10-RAG-Visuel-Embedding-Conjoint.ipynb), §2) encode images et textes dans **un même** espace vectoriel — c'est précisément ce que sa perte contrastive a appris à faire. La métrique du papier s'y lit directement. Pour chaque image *i* et sa légende *i* : est-ce que la légende est le **plus proche voisin** de l'image parmi les N textes (*fwd*) ? est-ce que l'image est la plus proche de sa légende parmi les N images (*bwd*) ? Le taux de paires où **les deux directions sont vraies à la fois** est le score d'alignement — les **voisins mutuels**.

In [3]:
MODELE_JOINT = "google/siglip-base-patch16-256-multilingual"
proc = AutoProcessor.from_pretrained(MODELE_JOINT)
joint = AutoModel.from_pretrained(MODELE_JOINT).to(DEVICE).eval()
print("Modele joint charge :", MODELE_JOINT)

@torch.no_grad()
def siglip_images(images):
    """Tour image -> vecteurs unitaires (768 dim)."""
    entrees = proc(images=images, return_tensors="pt").to(DEVICE)
    v = joint.get_image_features(pixel_values=entrees["pixel_values"])
    # transformers v5 : get_image_features renvoie un objet output, embedding poolé
    v = v.pooler_output.float().cpu().numpy()
    return v / np.linalg.norm(v, axis=1, keepdims=True)

@torch.no_grad()
def siglip_textes(textes):
    """Tour texte -> vecteurs unitaires (768 dim)."""
    entrees = proc(text=list(textes), padding=True, truncation=True,
                   return_tensors="pt").to(DEVICE)
    v = joint.get_text_features(**{k: t for k, t in entrees.items()
                                   if k in ("input_ids", "attention_mask")})
    v = v.pooler_output.float().cpu().numpy()
    return v / np.linalg.norm(v, axis=1, keepdims=True)

A = siglip_images([c["img"] for c in corpus])   # tour image -> (N, 768)
B = siglip_textes([c["cap"] for c in corpus])   # tour texte -> (N, 768)
print("Tour image :", A.shape, "  —  tour texte :", B.shape)

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.


[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

Modele joint charge : google/siglip-base-patch16-256-multilingual


Tour image : (72, 768)   —  tour texte : (72, 768)


In [4]:
def voisins_mutuels(A, B):
    """Taux de paires en voisins mutuels dans un espace commun.

    fwd = pour chaque image i, sa legende est le plus proche texte ;
    bwd = pour chaque legende i, son image est la plus proche image ;
    mut = les deux a la fois. Renvoie les trois taux (fractions du corpus).
    """
    sim = A @ B.T
    i2t = sim.argmax(1)          # texte le plus proche de chaque image
    t2i = sim.argmax(0)          # image la plus proche de chaque texte
    idx = np.arange(len(A))
    fwd = (i2t == idx).mean()
    bwd = (t2i == idx).mean()
    mut = ((i2t == idx) & (t2i == idx)).mean()
    return fwd, bwd, mut

fwd, bwd, mut = voisins_mutuels(A, B)
print(f"image -> sa legende                    : {fwd:.3f}")
print(f"legende -> son image                   : {bwd:.3f}")
print(f"VOISINS MUTUELS (score d'alignement)   : {mut:.3f}")

image -> sa legende                    : 0.069
legende -> son image                   : 0.042
VOISINS MUTUELS (score d'alignement)   : 0.014


### Lecture : le nombre mesuré, et pourquoi il surprend

Le score d'alignement est une **fraction du corpus** : la part de paires qui se reconnaissent mutuellement en premier voisin. La mesure donne **0,014** — soit 1/72, **le niveau du hasard** — et nettement **en dessous** du 0,16 que le papier obtient sur WIT (photos réelles, légendes humaines, **dans la distribution** d'entraînement).

C'est le résultat le plus instructif du carnet, et c'est le contraire de l'intuition « terrain synthétique = score élevé ». Un modèle d'alignement contrastif **ne transfère pas** son alignement cross-modal sur une distribution qu'il n'a jamais vue : face à 72 formes géométriques hors de sa distribution, SigLIP classe sa légende en premier voisin aussi souvent que par hasard. Le 0,16 de WIT est un chiffre **dans la distribution** ; on ne peut pas l'attendre **hors distribution**. Ce que mesure le §2, c'est donc une limite de **transfert**, pas un échec du modèle — et c'est précisément ce que le §5 (la géométrie CKA) et le §6 (le plafond) viennent éclairer de deux angles différents.

## 3. Le contrôle : une métrique vivante, un chiffre isolé muet

Avant d'interpréter, une vérification de bon sens : la métrique est-elle **vivante** ? On la met en défaut en brouillant l'appariement — chaque image est confrontée aux légendes *d'autres* images. Si la métrique mesure vraiment l'appariement, le score doit s'effondrer vers le hasard (1/N ≈ 0,014). Un score qui resterait élevé sur un appariement brouillé serait une métrique morte (saturée), pas une mesure.

Le second point est celui que le papier impose (§6) : le score absolu **dépend du terrain**. Élevé ici parce que les paires sont triviales ; bas sur WIT parce qu'elles ne le sont pas. Il ne se compare donc qu'**à lui-même** — entre modèles, entre échelles, entre niveaux d'information — jamais à un « bon niveau » imaginaire.

In [5]:
rng_perm = np.random.default_rng(SEED + 7)
perm = rng_perm.permutation(N)
_, _, mut_brouille = voisins_mutuels(A, B[perm])
print(f"Alignement sur appariement brouille : {mut_brouille:.3f}   (hasard ~ {1 / N:.3f})")

Alignement sur appariement brouille : 0.000   (hasard ~ 0.014)


### Lecture : le contrôle confirme que la métrique est vivante

Brouiller l'appariement fait passer le score de **0,014 à 0,000** : la métrique **réagit** à la structure des paires, elle n'est ni figée ni aveugle. C'est tout ce qu'on lui demande ici — répondre à l'appariement.

La nuance honnête : le score « réel » est **déjà** au niveau du hasard (0,014), donc l'écart avec le brouillé (0,000) est mince. Ce contrôle ne montre pas un grand fossé « aligné vs brouillé » — il montre que la métrique est **calculée et sensible**, et que le score bas du §2 n'est pas l'artefact d'une métrique morte mais un **résultat** (transfert hors distribution, §2 et §6). Le second point du papier tient quant à lui sans réserve : ce score ne se compare qu'**à lui-même** — entre encodeurs, entre granularités — jamais à un « bon niveau » imaginaire.

## 4. La pente intra-modale : l'alignement croît avec la taille de l'encodeur

Premier geste du papier : la convergence se lit **dans une même modalité**, entre modèles entraînés **indépendamment**. Le protocole ici : la tour texte de SigLIP fournit la représentation de référence des légendes (elle est alignée au visuel par entraînement). On encode les **mêmes** légendes avec des encodeurs texte autonomes — `e5-small` (33 M param., 384 dim) puis `e5-base` (278 M, 768 dim) — et l'on mesure la similarité représentationnelle par **CKA linéaire** (Kornblith, Noto, Isola 2019) : invariante aux rotations et au changement de dimension, elle compare la géométrie des deux ensembles de vecteurs à travers leurs matrices de Gram. Un plancher aléatoire (vecteurs gaussiens de même dimension) donne la valeur « aucune similarité ».

Si l'hypothèse tient, la CKA de l'encodeur autonome avec la référence **croît avec sa capacité** : plus un modèle texte est compétent, plus sa géométrie interne ressemble à celle de la tour alignée au visuel — sans que les deux modèles se soient jamais vus.

In [6]:
def cka(X, Y):
    """CKA lineaire : similarite de geometrie entre deux ensembles de vecteurs,
    invariante aux rotations et a la dimension (Kornblith et al. 2019)."""
    X = X - X.mean(0, keepdims=True)
    Y = Y - Y.mean(0, keepdims=True)
    KX = X @ X.T
    KY = Y @ Y.T
    num = (KX * KY).sum()
    den = np.sqrt((KX * KX).sum() * (KY * KY).sum())
    return float(num / den)

def e5_encoder(nom_modele):
    """Charge un encodeur e5 et renvoie textes -> vecteurs unitaires (moyenne poolée)."""
    tok = AutoTokenizer.from_pretrained(nom_modele)
    mod = AutoModel.from_pretrained(nom_modele).to(DEVICE).eval()
    @torch.no_grad()
    def encoder(textes):
        entrees = tok(list(textes), padding=True, truncation=True,
                      return_tensors="pt").to(DEVICE)
        s = mod(**entrees).last_hidden_state
        masque = entrees.attention_mask.unsqueeze(-1).float()
        v = (s * masque).sum(1) / masque.sum(1)
        v = v.float().cpu().numpy()
        return v / np.linalg.norm(v, axis=1, keepdims=True)
    return encoder

legendes = [c["cap"] for c in corpus]
E5_SMALL = "intfloat/multilingual-e5-small"
E5_BASE = "intfloat/multilingual-e5-base"
T_small = e5_encoder(E5_SMALL)(legendes)
T_base = e5_encoder(E5_BASE)(legendes)

rng_alea = np.random.default_rng(SEED + 11)
T_alea = rng_alea.standard_normal((N, T_base.shape[1]))

print("Reference : tour texte SigLIP (alignee au visuel)")
print(f"  CKA(reference, aleatoire {T_alea.shape[1]:>3}d) = {cka(B, T_alea):.3f}    <- plancher hasard")
print(f"  CKA(reference, e5-small {T_small.shape[1]:>3}d) = {cka(B, T_small):.3f}")
print(f"  CKA(reference, e5-base  {T_base.shape[1]:>3}d) = {cka(B, T_base):.3f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Reference : tour texte SigLIP (alignee au visuel)
  CKA(reference, aleatoire 768d) = 0.252    <- plancher hasard
  CKA(reference, e5-small 384d) = 0.624
  CKA(reference, e5-base  768d) = 0.522


### Lecture : la structure intra-modale est là — la pente, pas à cette échelle

Trois points sur l'axe de la compétence : hasard **0,252**, `e5-small` **0,624**, `e5-base` **0,522**. De cette sortie on lit deux choses, et l'une corrige l'intuition :

- **la géométrie intra-modale est bien réelle** : les deux encodeurs texte autonomes sont **loin** au-dessus du plancher aléatoire (0,62 et 0,52 contre 0,25) — leurs représentations textuelles partagent une structure, sans que ces modèles se soient jamais vus. C'est le signal de convergence **au sein d'une modalité** que le papier décrit.
- **la pente « plus compétent, plus aligné » ne se lit pas ici** : c'est le plus petit modèle qui est le plus proche de la référence (0,624 > 0,522), pas le plus grand. La pente de l'hypothèse platonicienne est un phénomène **d'échelle et de distribution** ; sur 72 légendes courtes **hors distribution**, à deux points seulement, elle n'apparaît pas. Ce que le carnet montre, c'est la **structure commune** intra-modale, pas sa croissance avec la capacité — celle-ci demanderait des modèles plus grands **et** un corpus dans la distribution.

La valeur absolue de chaque CKA ne se compare à rien d'universel ; c'est l'écart **au plancher aléatoire** qui porte l'information, et il est nettement positif pour les deux encodeurs.

## 5. Le test cross-modal : deux encodeurs jamais alignés

Le geste décisif du papier : mesurer l'alignement **entre modalités**, entre des modèles qui ne se sont **jamais** rencontrés. Ici : **`DINOv2-base`** (auto-supervision sur images seules — masquage/reconstruction, aucun texte dans l'entraînement) encode les images ; **`e5-base`** (texte seul) encode les légendes. Aucun entraînement conjoint, aucune perte contrastive, aucun espace commun : si leurs représentations s'alignent, c'est une convergence **spontée**. Le témoin est la paire jointe de SigLIP, alignée **par construction**.

Le tableau croise les deux par CKA : la paire construite contre la paire indépendante. (Le CKA est l'outil adapté ici : contrairement au mutual NN du §2, il compare deux espaces **différents** — pas besoin qu'ils soient déjà communs.)

In [7]:
MODELE_DINO = "facebook/dinov2-base"
dino_proc = AutoImageProcessor.from_pretrained(MODELE_DINO)
dino = AutoModel.from_pretrained(MODELE_DINO).to(DEVICE).eval()
print("Modele image independant charge :", MODELE_DINO)

@torch.no_grad()
def dino_images(images, taille_lot=24):
    """DINOv2 : moyenne des tokens de patch (sans le CLS), vecteurs unitaires."""
    lots = []
    for debut in range(0, len(images), taille_lot):
        entrees = dino_proc(images=images[debut:debut + taille_lot],
                            return_tensors="pt").to(DEVICE)
        lots.append(dino(**entrees).last_hidden_state[:, 1:, :].mean(1))
    v = torch.cat(lots).float().cpu().numpy()
    return v / np.linalg.norm(v, axis=1, keepdims=True)

D = dino_images([c["img"] for c in corpus])
print("DINOv2 images :", D.shape)

print()
print(f"CKA paire CONSTRUITE   (SigLIP image,  SigLIP texte) : {cka(A, B):.3f}")
print(f"CKA paire INDEPENDANTE (DINOv2 image,  e5 texte)     : {cka(D, T_base):.3f}")
print(f"(reference hasard mesuré au §4 : {cka(B, T_alea):.3f})")

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Modele image independant charge : facebook/dinov2-base


DINOv2 images : (72, 768)

CKA paire CONSTRUITE   (SigLIP image,  SigLIP texte) : 0.471
CKA paire INDEPENDANTE (DINOv2 image,  e5 texte)     : 0.617
(reference hasard mesuré au §4 : 0.252)


### Lecture : la surprise — et ce qu'elle dit de la métrique

La sortie contredit l'attente naïve : la paire **indépendante** (DINOv2 image + e5 texte, **0,617**) est **plus** alignée par CKA que la paire **construite** de SigLIP (**0,471**). L'alignement cross-modal n'est **pas** ce que la perte contrastive « achète », du point de vue de la CKA — c'est l'inverse.

La raison est éclairante et concerne la **métrique** plus que les modèles. La CKA linéaire compare la **géométrie** des représentations — la structure de variance commune — et non la **récupération** (mesurée par les voisins mutuels du §2). Sur ce corpus, quelques attributs (couleur, forme, nombre, taille) dominent **toute** la variance. Deux encodeurs forts, chacun dans sa modalité, extraient **indépendamment** ces mêmes axes dominants : leurs géométries se ressemblent donc (CKA élevée), **alors même que ni l'un ni l'autre ne peut récupérer la modalité de l'autre** (voisins mutuels au hasard, §2). Entraîner un espace **joint** (SigLIP), qui pousse au contraire à **séparer** les paires, peut abaisser la CKA sur un petit échantillon.

C'est la leçon la plus utile du carnet : **CKA et voisins mutuels mesurent des choses différentes, et ils s'opposent ici** — géométrie partagée élevée, récupération nulle. L'hypothèse platonicienne est subtile et **sensible à la métrique** ; à cette échelle, hors distribution, la « convergence » se lit en géométrie commune, pas en alignement exploitable. C'est exactement le type de nuance qui fait la valeur du travail : poser la thèse **en termes mesurables**, et montrer que la mesure dépend de l'instrument.

## 6. Le plafond d'information mutuelle : légendes longue / moyenne / courte

Le papier borne l'alignement par l'**information mutuelle** du couple image/légende (Figure 9) : une légende qui dit peu ne peut pas être retrouvée parmi beaucoup. Le corpus s'y prête exactement, parce que la légende encode un nombre variable de degrés de liberté :

- **complète** : forme + couleur + nombre + taille → 3 × 4 × 3 × 2 = **72** classes distinctes ;
- **moyenne** : forme + couleur + nombre (taille retirée) → 3 × 4 × 3 = **36** classes ;
- **courte** : forme + couleur (nombre et taille retirés) → 3 × 4 = **12** classes.

Le plafond **combinatoire** du taux de voisins mutuels est alors connu d'avance : au mieux `classes distinctes / N` — plusieurs images partagent la même légende, une seule peut être leur voisine mutuelle gagnante. Mesurer le score à chaque niveau sépare proprement ce que le **modèle** sait faire de ce que le **signal** permet — et rend visible le plafond que le papier invoque.

In [8]:
def legende_moyenne(c):
    return f"{NOMBRES[c['n']]} {pluriel(c['n'], NOM_FORME[c['forme']])} {pluriel(c['n'], c['couleur'])}"

def legende_courte(c):
    return f"{NOM_FORME[c['forme']]} {c['couleur']}"

niveaux = {
    "complete (forme+couleur+nombre+taille)": [c["cap"] for c in corpus],
    "moyenne  (forme+couleur+nombre)       ": [legende_moyenne(c) for c in corpus],
    "courte   (forme+couleur)              ": [legende_courte(c) for c in corpus],
}
print(f"{'niveau de legende':<40}{'classes':>8}{'plafond':>9}{'mesure':>9}")
for nom, legends in niveaux.items():
    classes = len(set(legends))
    plafond = classes / N
    _, _, score = voisins_mutuels(A, siglip_textes(legends))
    print(f"{nom:<40}{classes:>8}{plafond:>9.3f}{score:>9.3f}")

niveau de legende                        classes  plafond   mesure
complete (forme+couleur+nombre+taille)        72    1.000    0.014
moyenne  (forme+couleur+nombre)               36    0.500    0.000
courte   (forme+couleur)                      12    0.167    0.000


### Lecture : le modèle reste au plancher — le plafond est réel mais non atteint

Le score reste **au niveau du hasard** à chaque granularité — **0,014** (complète), **0,000** (moyenne), **0,000** (courte) — alors que le **plafond combinatoire** monte (1,00 / 0,50 / 0,17). Il n'y a donc **pas de remontée** vers le plafond : la légende ne devient pas plus récupérable quand elle dit plus de choses, parce que le modèle, **hors distribution**, ne sort pas du hasard à **aucun** niveau de détail.

Ce que la section établit quand même, et c'est le lien avec le papier : le plafond par **information mutuelle** est une borne **réelle et calculable** d'avance (ici `classes / N`) — le signal ne peut pas être mieux récupéré que ce qu'il contient. Mais sur ce corpus hors distribution, c'est le **plancher** (le hasard) qui fait le plafond **effectif** : le modèle n'atteint pas le plafond théorique. La limite d'alignement est donc double : ce que le **signal** permet (le plafond, § du papier) et ce que le **modèle** transfère sur une distribution vue ou non (ici : rien au-delà du hasard). Le couple « plafond théorique connu, score au plancher » rend cette double limite **visible**.

## 7. Exercices

Trois prolongements, du plus mécanique au plus expérimental.

### Exercice 1 — Étendre la pente d'un troisième point

La pente du §4 ne repose que sur deux encodeurs. Chargez `intfloat/multilingual-e5-large` (560 M, 1024 dim — téléchargement ≈ 2,2 Go), ou tout autre encodeur texte à votre disposition, encodez les **mêmes** légendes et ajoutez le point : la CKA avec la référence SigLIP continue-t-elle de monter ? L'écart `e5-base → e5-large` est-il **plus petit** que `e5-small → e5-base` (rendements décroissants) ?

In [9]:
# Exercice 1 -- a completer
# Indice : e5_encoder("intfloat/multilingual-e5-large") donne l'encodeur ;
# encoder(legendes) les vecteurs ; cka(B, T_large) le nouveau point.
print("Exercice a completer : troisieme point de la pente (e5-large)")

Exercice a completer : troisieme point de la pente (e5-large)


### Exercice 2 — Recoudre deux espaces : le model stitching

Moschella, Sarti et al. (2022, « Stitching Neural Networks », cité par le papier) montrent que des réseaux entraînés séparément peuvent se **recoudre** par une simple couche linéaire. Implémentez-le : sur 60 % des paires, ajustez une régression linéaire (Ridge) qui projette les vecteurs DINOv2 vers les vecteurs `e5` ; projetez ensuite les 40 % restants, et mesurez le taux de voisins mutuels **après recousure**. Une couche affine suffit-elle à donner aux deux espaces indépendants du §5 un alignement **utilisable** ? (Renvoyez le taux de la recousure et le comparez au hasard 1/n_test.)

In [10]:
# Exercice 2 -- a completer
# Indice : W = ridge(D_entrainement, T_base_entrainement) ; D_test @ W donne
# l'image dans l'espace e5 ; puis voisins_mutuels(D_test @ W, T_base_test).
print("Exercice a completer : recousure lineaire DINOv2 -> e5 (model stitching)")

Exercice a completer : recousure lineaire DINOv2 -> e5 (model stitching)


### Exercice 3 — Le plafond était-il le modèle ou le signal ?

Le §6 plafonne la légende courte à 12 classes pour N = 72 (plafond ≈ 0,17). Faites
varier le ratio **classes / N** : filtrez le corpus sur une seule taille (par exemple
`c["taille"] == "petit"`) — on tombe à 36 images pour 12 classes courtes, plafond
≈ 0,33 — puis sur une seule couleur. Le taux de voisins mutuels de la légende courte
**remonte-t-il** vers le nouveau plafond, à modèle constant ? Concluez en une phrase :
à information et modèle fixes, que fait le ratio classes/N au score ?

In [11]:
# Exercice 3 -- a completer
# Indice : sous = [c for c in corpus if c["taille"] == "petit"] ; encoder images
# (A) et legendes courtes (B) sur `sous`, puis voisins_mutuels ; comparer au plafond 12/36.
print("Exercice a completer : ratio classes/N et plafond du score")

Exercice a completer : ratio classes/N et plafond du score


## Conclusion

Ce carnet a mesuré, sur un corpus déterministe **hors distribution**, les trois gestes du papier — et la lecture honnête est celle des **surprises**, pas des confirmations.

1. **La métrique est vivante, le score est au hasard** (§2, §3) : les voisins mutuels de SigLIP sur 72 formes synthétiques sont à **0,014 = 1/N**, le niveau du hasard, et brouiller l'appariement les fait passer à 0. L'alignement contrastif **ne transfère pas** sur une distribution non vue — le 0,16 de WIT est un chiffre **dans la distribution**.
2. **La géométrie intra-modale est réelle, la pente ne se lit pas à cette échelle** (§4) : `e5-small` (0,624) et `e5-base` (0,522) sont tous deux **loin** au-dessus du hasard (0,252) — la structure commune des représentations texte existe — mais le plus petit modèle est le plus aligné ; la pente « capacité » du papier n'apparaît pas sur 72 légendes courtes hors distribution.
3. **CKA et voisins mutuels s'opposent** (§5) : la paire **indépendante** DINOv2+e5 (0,617) est plus alignée par **CKA** que la paire **jointe** SigLIP (0,471), alors que la récupération (voisins mutuels) est au hasard. La CKA mesure la **géométrie commune** (les deux encodeurs extraient indépendamment les mêmes attributs dominants), pas l'**alignement exploitable** — la thèse est **sensible à la métrique**.
4. **Le plafond d'information est réel, le modèle reste au plancher** (§6) : le plafond combinatoire monte avec l'informativité (1,00 / 0,50 / 0,17) mais le score reste au hasard à chaque niveau — la limite d'alignement est double : ce que le signal **permet** (le plafond) et ce que le modèle **transfère** hors distribution (ici : rien).

L'apport du carnet n'est pas de « confirmer » l'hypothèse platonicienne — c'est de **montrer, sur des mesures réelles, comment elle se pose en termes mesurables et où elle se complique** : transfert hors distribution, sensibilité à la métrique (CKA vs récupération), plafond par information. La suite naturelle : refaire ces mesures **dans la distribution** (photos + légendes naturelles) et à **plus grande échelle**, là où le papier observe la convergence — et y comparer CKA et voisins mutuels.

*Note finale — la question de Molyneux (1688) : un aveugle-né à qui l'on rend la vue reconnaît-il un cube qu'il connaît au toucher ? L'hypothèse platonicienne en est la version moderne et mesurable : si les représentations du toucher et de la vue convergent vers le même modèle statistique du monde, la réponse est oui. Ce carnet la pose **en termes mesurables** — et montre, sur des formes et des légendes, que la mesure dépend de la distribution **et** de l'instrument. Il ne la tranche pas.*